# Hakem deneyleri - Round 2 (reviewer_round2_v1)

30 yeni egitim (3 seed): bagimsiz Teacher 2 (Hakem 1.6/2.3), spectral-norm'suz Teacher 2 (1.8),
tum asamalarda augmentation (2.2), Fusion CNN-ViT yeniden egitimi (1.5); ayrica gated/equal vs CE
eslenik guven araliklari, etiket butcesi araliklari, IDRiD/Messidor-2/DDR dis test ve gated Grad-CAM (2.5).

GPU secin. Kesinti olursa ayni RUN_ID ile kurulum hucrelerini yeniden calistirip egitim hucresine devam edin.
Round-1 klasorunu (`runs/reviewer_complete_v1`) SILMEYIN; sadece okunur. Bulunamazsa referanslar yeniden egitilir.
Drive'da en az **30 GiB** bos alan planlayin. Sonuclarin olumlu olacagi varsayilmaz.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT = Path('/content/drive/MyDrive/DR_Reviewer_Complete')
RUN_ID = 'reviewer_round2_v1'
ROUND1 = PROJECT / 'runs' / 'reviewer_complete_v1'   # read-only
FAMILIES = None          # None: all; e.g. ['t2indep'] or ['fusion', 'nosn']
JOBS_THIS_SESSION = None # None: all unfinished jobs; e.g. 2
STORAGE_CONFIRMED = False
# Optional external datasets (leave None if not available):
MESSIDOR2_IMAGES = None  # e.g. PROJECT / 'messidor-2.zip'
MESSIDOR2_LABELS = None  # e.g. PROJECT / 'messidor_data.csv'  (Krause adjudicated)
DDR_IMAGES = None        # e.g. PROJECT / 'DDR_test_images.zip'
DDR_LABELS = None        # e.g. PROJECT / 'test.txt'
if not STORAGE_CONFIRMED:
    raise RuntimeError('Drive hesabinda en az 30 GiB bos alan oldugunu kontrol edip STORAGE_CONFIRMED=True yapin.')
CODE_ZIP = PROJECT / 'DR_Reviewer_Round2_Colab.zip'
DATA_ZIP = PROJECT / 'DR_Reviewer_Complete_Data.zip'
IDRID_ZIP = PROJECT / 'B. Disease Grading.zip'
for path in (CODE_ZIP, DATA_ZIP, IDRID_ZIP):
    if not path.is_file():
        raise FileNotFoundError(path)
print('Round-1 klasoru var mi:', ROUND1.is_dir())
REMOTE = PROJECT / 'runs' / RUN_ID
LOCAL = Path('/content/reviewer_runs') / RUN_ID
REMOTE.mkdir(parents=True, exist_ok=True)
LOCAL.mkdir(parents=True, exist_ok=True)

In [ ]:
import hashlib, json, subprocess, sys, zipfile
from pathlib import PurePosixPath
WORK = Path('/content/reviewer_workspace')
WORK.mkdir(exist_ok=True)
def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()
def unpack(path, destination):
    with zipfile.ZipFile(path) as archive:
        for name in archive.namelist():
            member = PurePosixPath(name)
            if member.is_absolute() or '..' in member.parts or chr(92) in name:
                raise ValueError('Unsafe/nonportable ZIP member: ' + name)
        archive.extractall(destination)
unpack(CODE_ZIP, WORK)
PACKAGE = WORK / 'colab_reviewer_round2'
manifest = json.loads((PACKAGE / 'MANIFEST.json').read_text())
for name, expected in manifest.items():
    if sha256(PACKAGE / name) != expected:
        raise ValueError('Package hash mismatch: ' + name)
DATA = WORK / 'APTOS2019'
if not (DATA / 'cache_224.npz').exists():
    unpack(DATA_ZIP, WORK)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PACKAGE / 'requirements.txt')], check=True)
import torch, torchvision, timm
print('Torch:', torch.__version__, 'torchvision:', torchvision.__version__, 'timm:', timm.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('Colab GPU runtime secilmeli.')
print('GPU:', torch.cuda.get_device_name(0))
import os, time
LOGS = LOCAL / 'logs'
LOGS.mkdir(exist_ok=True)
def execute(script, *arguments):
    # Streams child output line by line into the cell and into LOCAL/logs (also copied to Drive).
    command = [sys.executable, '-u', str(PACKAGE / script), *map(str, arguments)]
    env = dict(os.environ, PYTHONUNBUFFERED='1')
    log = LOGS / (Path(script).stem + '.log')
    started = time.time()
    with log.open('a', encoding='utf-8') as stream:
        stream.write(f'\n===== {time.ctime()} {command}\n')
        process = subprocess.Popen(command, cwd=PACKAGE, env=env, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                stream.write(line)
                stream.flush()
            code = process.wait()
        except KeyboardInterrupt:
            process.kill()
            raise
    (REMOTE / 'logs').mkdir(exist_ok=True)
    subprocess.run(['cp', str(log), str(REMOTE / 'logs' / log.name)], check=False)
    print(f'[{script} bitti: {time.time() - started:.0f} s, kod {code}]', flush=True)
    if code != 0:
        raise subprocess.CalledProcessError(code, command)
COMMON = ['--data', DATA, '--output', LOCAL, '--drive', REMOTE]
ROUND1_ARGS = ['--round1', ROUND1] if ROUND1.is_dir() else []
FAMILY_ARGS = [] if FAMILIES is None else ['--families', *FAMILIES]

## Kontroller ve envanter
Testler/smoke deney sonucu degildir. Envanter, round-1 checkpoint ve npz dosyalarinin Drive'da olup olmadigini
yazar ve varsa referanslari SHA-256 dogrulamasiyla kopyalar (`reference_manifest.json`: imported / retrained).

In [ ]:
execute('test_round2.py')
execute('test_experiments.py')
execute('smoke2.py', '--pretrained')
execute('run2.py', '--stage', 'inventory', *COMMON, *ROUND1_ARGS, *FAMILY_ARGS)
print(json.dumps(json.loads((LOCAL / 'round1_inventory.json').read_text())['counts'], indent=1))

## Egitim / kesintiden devam
Hucreyi tekrar calistirmak biten isleri atlar. Secim validasyon QWK ile yapilir; test kullanilmaz.

In [ ]:
limit = [] if JOBS_THIS_SESSION is None else ['--limit', JOBS_THIS_SESSION]
execute('run2.py', '--stage', 'train', *COMMON, *ROUND1_ARGS, *FAMILY_ARGS, *limit)
sys.path.insert(0, str(PACKAGE))
from plan2 import jobs
wanted = [j for j in jobs() if FAMILIES is None or j.family in set(FAMILIES) | {'reference'}]
done = [j.key for j in wanted if (LOCAL / j.key / 'complete.json').exists()]
print(f'Tamamlanan: {len(done)}/{len(wanted)}')
if len(done) < len(wanted):
    raise RuntimeError('Egitim bitmedi. Bu hucreyi tekrar calistirin; test/analize gecmeyin.')

## Test degerlendirmesi ve analiz
Sabit protokol: per-seed metrikler, mean/SD, validasyonla sicaklik kalibrasyonu, 10.000 eslenik bootstrap
(gated/equal vs bagimsiz CE; yeni kollar vs referans) ve round-1 etiket butcesi araliklari.

In [ ]:
execute('run2.py', '--stage', 'evaluate', *COMMON, *ROUND1_ARGS, *FAMILY_ARGS)
execute('analyze2.py', '--output', LOCAL, '--drive', REMOTE, *ROUND1_ARGS)

## Dis veri setleri ve gated Grad-CAM
IDRiD 103 resmi test; istege bagli Messidor-2 (derecelendirilebilir) ve DDR test (derece 5 haric).
Hedef veride egitim/kalibrasyon/esik ayari yapilmaz. Grad-CAM nitel gosterimdir, lezyon dogrulamasi degildir.

In [ ]:
extra = []
if MESSIDOR2_IMAGES and MESSIDOR2_LABELS:
    extra += ['--extra', 'Messidor2', MESSIDOR2_IMAGES, MESSIDOR2_LABELS, 'messidor2']
if DDR_IMAGES and DDR_LABELS:
    extra += ['--extra', 'DDR', DDR_IMAGES, DDR_LABELS, 'ddr']
execute('external2.py', '--zip', IDRID_ZIP, *COMMON, *extra)

## Sonuc paketini indir
Checkpoint (.pth) icermez; analiz, dis test, manifestler ve tum logit/metrik dosyalari.

In [ ]:
import shutil
OUT = PROJECT / f'{RUN_ID}_results.zip'
with zipfile.ZipFile('/content/results.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(LOCAL.rglob('*')):
        if path.is_file() and path.suffix in {'.json', '.npz', '.md', '.png', '.csv'}:
            archive.write(path, path.relative_to(LOCAL).as_posix())
shutil.copy('/content/results.zip', OUT)
print('Sonuc ZIP:', OUT, round(OUT.stat().st_size / 2**20, 1), 'MiB')
from google.colab import files
files.download('/content/results.zip')